In [1]:
# CELL 1: Imports and device

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModel
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 2: Load FE training and test CSV files

train_path = "/content/drive/MyDrive/SemEval-task-6-main/csv_files/FE_training_data"
test_path  = "/content/drive/MyDrive/SemEval-task-6-main/csv_files/FE_test_data"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

train_df.head(), test_df.head()


(   Unnamed: 0                                            title  \
 0           0  The President's News Conference in Osaka, Japan   
 1           1                  The President's News Conference   
 2           2                  The President's News Conference   
 3           3                  The President's News Conference   
 4           4                  The President's News Conference   
 
                  date        president  \
 0       June 29, 2019  Donald J. Trump   
 1  September 20, 2007   George W. Bush   
 2   December 20, 2007   George W. Bush   
 3   November 14, 2012     Barack Obama   
 4    January 18, 2017     Barack Obama   
 
                                                  url  question_order  \
 0  https://www.presidency.ucsb.edu/documents/the-...              33   
 1  https://www.presidency.ucsb.edu/documents/the-...              13   
 2  https://www.presidency.ucsb.edu/documents/the-...               4   
 3  https://www.presidency.ucsb.edu/document

In [4]:
# CELL 3: Label mappings for Task 1 (clarity: 3 classes)

assert "clarity_label" in train_df.columns, train_df.columns

labels_t1 = sorted(train_df["clarity_label"].unique().tolist())
label2id_t1 = {lab: i for i, lab in enumerate(labels_t1)}
id2label_t1 = {i: lab for lab, i in label2id_t1.items()}
num_labels_t1 = len(labels_t1)

label2id_t1, id2label_t1


({'Ambivalent': 0, 'Clear Non-Reply': 1, 'Clear Reply': 2},
 {0: 'Ambivalent', 1: 'Clear Non-Reply', 2: 'Clear Reply'})

In [5]:
# CELL 4: Train / validation split

train_df_t1, valid_df_t1 = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["clarity_label"]
)

train_df_t1.shape, valid_df_t1.shape, test_df.shape


((2344, 27), (414, 27), (690, 27))

In [6]:
# CELL 5: Tokenizer and feature column names

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# These should match your FE_* columns.
feature_cols = ["q_word_count", "a_word_count", "a_negation_count", "qa_cosine_sim"]
for col in feature_cols:
    assert col in train_df.columns, f"Missing column: {col}"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
# CELL 6: Tokenization with features for Task 1

def tokenize_and_add_features_task1(batch):
    # Tokenize Q/A pair
    enc = tokenizer(
        batch["question"],
        batch["interview_answer"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    feats = []
    for q_wc, a_wc, a_neg, qa_sim in zip(
        batch["q_word_count"],
        batch["a_word_count"],
        batch["a_negation_count"],
        batch["qa_cosine_sim"],
    ):
        feats.append([q_wc, a_wc, a_neg, qa_sim])

    enc["features"] = feats

    # map clarity labels to ids
    enc["labels"] = [label2id_t1[x] for x in batch["clarity_label"]]

    return enc


In [8]:
# CELL 7: Build Datasets and apply tokenization

train_ds_t1 = Dataset.from_pandas(train_df_t1.reset_index(drop=True))
valid_ds_t1 = Dataset.from_pandas(valid_df_t1.reset_index(drop=True))
test_ds_t1  = Dataset.from_pandas(test_df.reset_index(drop=True))

train_enc_t1 = train_ds_t1.map(
    tokenize_and_add_features_task1,
    batched=True,
    remove_columns=train_ds_t1.column_names,
)

valid_enc_t1 = valid_ds_t1.map(
    tokenize_and_add_features_task1,
    batched=True,
    remove_columns=valid_ds_t1.column_names,
)

test_enc_t1 = test_ds_t1.map(
    tokenize_and_add_features_task1,
    batched=True,
    remove_columns=test_ds_t1.column_names,
)

train_enc_t1.set_format(type="torch")
valid_enc_t1.set_format(type="torch")
test_enc_t1.set_format(type="torch")

train_enc_t1[0]


Map:   0%|          | 0/2344 [00:00<?, ? examples/s]

Map:   0%|          | 0/414 [00:00<?, ? examples/s]

Map:   0%|          | 0/690 [00:00<?, ? examples/s]

{'input_ids': tensor([    0, 22261,     9,     5,   499,  3839,     9,     5,  5490,    12,
           791,     4,   104,     4,  1291,     4,     2,     2, 13987,    47,
            13,   110,   864,     4,  2647,     6,   148,     5,  1431,    14,
            38,    56,    19,     5,   270,     6,    52,  5055,    14,    89,
            16,   202,   203,   929,    13,   617,  2919,     9,     5,  9526,
          3115,     6,     8,    52,  3373,  1319,     8,   839,     7,   617,
          5920,     5,  4601,    11,    10,    55, 24821,     8,  2375,  4737,
            11,     5,    86,     7,   283,     4,   178,     5,    80,  2380,
            67,  7114,     7,   712,  9872,     8, 25730,  3663,     6,   941,
             5,   239,    12,  4483,  2891,   149,  9526,  5695,     8,  2891,
            23,     5, 10151,     9,     5,  2174,     8,   758, 20678,     4,
            20,    80,  2380,    40,    67,  3720,     5,  4093,    13,   709,
             9,     5,   776,     6,   

In [9]:
# CELL 8: Model definition – RoBERTa + 4 numeric features

class RobertaWithTabularFeatures(nn.Module):
    def __init__(self, model_name, num_labels, feature_dim):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size + feature_dim, num_labels)

    def forward(self, input_ids, attention_mask, features, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:, 0]  # RoBERTa's [CLS]-like token

        x = torch.cat([pooled, features], dim=-1)
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


In [10]:
# ==== MODEL CELL FOR TASK 1 – RUN THIS BEFORE TRAINER ====

import torch.nn as nn
from transformers import AutoModel

# We assume these already exist from previous cells:
# - model_name = "roberta-base"
# - feature_cols = ["q_word_count", "a_word_count", "a_negation_count", "qa_cosine_sim"]
# - num_labels_t1, id2label_t1, label2id_t1
# - device

class RobertaWithTabularFeatures(nn.Module):
    def __init__(self, model_name, num_labels, feature_dim):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size + feature_dim, num_labels)

    def forward(self, input_ids, attention_mask, features, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # [CLS]-like summary token
        pooled = outputs.last_hidden_state[:, 0]

        x = torch.cat([pooled, features], dim=-1)
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


# 👇 THIS LINE CREATES THE MODEL FOR TASK 1
model_t1 = RobertaWithTabularFeatures(
    model_name=model_name,
    num_labels=num_labels_t1,
    feature_dim=len(feature_cols),
).to(device)

print("Model ready:", model_t1.__class__.__name__)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model ready: RobertaWithTabularFeatures


In [11]:
# CELL 9: Metrics & TrainingArguments & Trainer for Task 1 (FIXED for your HF version)

def compute_metrics_task1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
    }


training_args_t1 = TrainingArguments(
    output_dir="out/task1-roberta-FE",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
)


trainer_t1 = Trainer(
    model=model_t1,
    args=training_args_t1,
    train_dataset=train_enc_t1,
    eval_dataset=valid_enc_t1,
    compute_metrics=compute_metrics_task1,
)


In [12]:
# CELL 10: Train Task 1 RoBERTa + features

trainer_t1.train()


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Weighted F1
1,4.085700,4.470382,0.243797,0.350242,0.317053
2,2.796200,3.804391,0.285901,0.410628,0.398687
3,2.693300,3.070477,0.416697,0.487923,0.490251


TrainOutput(global_step=879, training_loss=3.7603572719604355, metrics={'train_runtime': 458.5585, 'train_samples_per_second': 15.335, 'train_steps_per_second': 1.917, 'total_flos': 0.0, 'train_loss': 3.7603572719604355, 'epoch': 3.0})

In [13]:
# CELL 11: Evaluate on test set and print classification report

preds_t1 = trainer_t1.predict(test_enc_t1)

y_true_t1 = preds_t1.label_ids
y_pred_t1 = preds_t1.predictions.argmax(axis=-1)

print("=== Task 1 RoBERTa + FE – Test performance (FE_test_data) ===")
print(classification_report(
    y_true_t1,
    y_pred_t1,
    target_names=[id2label_t1[i] for i in range(num_labels_t1)],
))


=== Task 1 RoBERTa + FE – Test performance (FE_test_data) ===
                 precision    recall  f1-score   support

     Ambivalent       0.66      0.53      0.59       408
Clear Non-Reply       0.57      0.44      0.50        71
    Clear Reply       0.37      0.54      0.44       211

       accuracy                           0.52       690
      macro avg       0.53      0.50      0.51       690
   weighted avg       0.56      0.52      0.53       690



In [14]:
train_df.columns


Index(['Unnamed: 0', 'title', 'date', 'president', 'url', 'question_order',
       'interview_question', 'interview_answer', 'gpt3.5_summary',
       'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1',
       'annotator2', 'annotator3', 'inaudible', 'multiple_questions',
       'affirmative_questions', 'index', 'clarity_label', 'evasion_label',
       'clarity_label_id', 'evasion_label_id', 'q_word_count', 'a_word_count',
       'a_negation_count', 'qa_cosine_sim'],
      dtype='object')

In [15]:
# CELL T2-1: Define which column is the evasion label column
TASK2_LABEL_COL = "evasion_label"  # ⬅️ CHANGE THIS if your column name is different

assert TASK2_LABEL_COL in train_df.columns, train_df.columns


In [16]:
# CELL T2-2: Build label mappings for Task 2 (9 evasion classes)

labels_t2 = sorted(train_df[TASK2_LABEL_COL].unique().tolist())
label2id_t2 = {lab: i for i, lab in enumerate(labels_t2)}
id2label_t2 = {i: lab for lab, i in label2id_t2.items()}
num_labels_t2 = len(labels_t2)

label2id_t2, num_labels_t2


({'Claims ignorance': 0,
  'Clarification': 1,
  'Declining to answer': 2,
  'Deflection': 3,
  'Dodging': 4,
  'Explicit': 5,
  'General': 6,
  'Implicit': 7,
  'Partial/half-answer': 8},
 9)

In [17]:
# CELL T2-3: Train/validation split for Task 2

train_df_t2, valid_df_t2 = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df[TASK2_LABEL_COL]
)

train_df_t2.shape, valid_df_t2.shape, test_df.shape


((2344, 27), (414, 27), (690, 27))

In [18]:
# CELL T2-4: Build HuggingFace Datasets for Task 2

train_ds_t2 = Dataset.from_pandas(train_df_t2.reset_index(drop=True))
valid_ds_t2 = Dataset.from_pandas(valid_df_t2.reset_index(drop=True))
test_ds_t2  = Dataset.from_pandas(test_df.reset_index(drop=True))

train_ds_t2, valid_ds_t2, test_ds_t2


(Dataset({
     features: ['Unnamed: 0', 'title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label', 'clarity_label_id', 'evasion_label_id', 'q_word_count', 'a_word_count', 'a_negation_count', 'qa_cosine_sim'],
     num_rows: 2344
 }),
 Dataset({
     features: ['Unnamed: 0', 'title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label', 'clarity_label_id', 'evasion_label_id', 'q_word_count', 'a_word_count', 'a_negation_count', 'qa_cosine_sim'],
     num_rows: 414
 }),
 Dataset({
     features: ['Unn

In [19]:
# CELL T2-5: Tokenization + features for Task 2

def tokenize_and_add_features_task2(batch):
    # Tokenize question + answer pair
    enc = tokenizer(
        batch["question"],
        batch["interview_answer"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    # Reuse the same 4 numeric features
    feats = []
    for q_wc, a_wc, a_neg, qa_sim in zip(
        batch["q_word_count"],
        batch["a_word_count"],
        batch["a_negation_count"],
        batch["qa_cosine_sim"],
    ):
        feats.append([q_wc, a_wc, a_neg, qa_sim])

    enc["features"] = feats

    # Map evasion labels → ids
    enc["labels"] = [label2id_t2[x] for x in batch[TASK2_LABEL_COL]]

    return enc

train_enc_t2 = train_ds_t2.map(
    tokenize_and_add_features_task2,
    batched=True,
    remove_columns=train_ds_t2.column_names,
)

valid_enc_t2 = valid_ds_t2.map(
    tokenize_and_add_features_task2,
    batched=True,
    remove_columns=valid_ds_t2.column_names,
)

test_enc_t2 = test_ds_t2.map(
    tokenize_and_add_features_task2,
    batched=True,
    remove_columns=test_ds_t2.column_names,
)

train_enc_t2.set_format(type="torch")
valid_enc_t2.set_format(type="torch")
test_enc_t2.set_format(type="torch")

train_enc_t2[0]


Map:   0%|          | 0/2344 [00:00<?, ? examples/s]

Map:   0%|          | 0/414 [00:00<?, ? examples/s]

Map:   0%|          | 0/690 [00:00<?, ? examples/s]

{'input_ids': tensor([    0, 13755,    47,  8578,    19,  1869, 12974,  1292,   926, 37252,
          1996,   320,  1321,    23,    39,   940,   138,     7,   146,  5215,
             7,     5,  3932,     8,   172, 27736,   106,   116,     2,     2,
          9904,     6,    38,   218,    75,   216,   350,   203,    59,    24,
             4,    38,  1166,   402,    42,   662,     6,    53,    38,   218,
            75,   578,  7443,    87,    14,     6,    38,  1017,    33,     7,
           192,    24,     4,    91,    18,    10,   182,  9132,   313,     4,
            91,    21,  2033,   182,   203,    30,   258,  1799,     6,    38,
          4443,     4,    85,    21,  2345,     9,    41,  2846,    14,   362,
           317,    30,   258,  1799,     4,    38,   218,    75,   216,  2230,
            99,     5,   527,    16,     4,    38,   581,  1819,   216,   624,
            10,   765,   675,     9,    86,     4,    38,    95,  1166,    24,
            13,     5,    78,    86,   

In [20]:
# CELL T2-6: Initialize Task 2 model (same architecture, 9 labels)

model_t2 = RobertaWithTabularFeatures(
    model_name=model_name,           # "roberta-base"
    num_labels=num_labels_t2,        # 9
    feature_dim=len(feature_cols),   # 4
).to(device)

print("Task 2 model ready:", model_t2.__class__.__name__, "| num_labels:", num_labels_t2)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Task 2 model ready: RobertaWithTabularFeatures | num_labels: 9


In [21]:
# CELL T2-7: Metrics + TrainingArguments + Trainer for Task 2

def compute_metrics_task2(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
    }

training_args_t2 = TrainingArguments(
    output_dir="out/task2-roberta-FE",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,

    eval_strategy="epoch",        # your HF version uses eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
)

trainer_t2 = Trainer(
    model=model_t2,
    args=training_args_t2,
    train_dataset=train_enc_t2,
    eval_dataset=valid_enc_t2,
    compute_metrics=compute_metrics_task2,
)


In [22]:
# CELL T2-8: Train Task 2 RoBERTa + features

trainer_t2.train()


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Weighted F1
1,3.715300,3.238001,0.087147,0.263285,0.179122
2,3.560900,2.886460,0.078383,0.299517,0.183108
3,3.186600,2.679154,0.142263,0.243961,0.212236
4,2.411600,2.591198,0.258775,0.326087,0.276891
5,2.367000,2.520463,0.249992,0.333333,0.267191


TrainOutput(global_step=1465, training_loss=3.313075066263765, metrics={'train_runtime': 732.0212, 'train_samples_per_second': 16.01, 'train_steps_per_second': 2.001, 'total_flos': 0.0, 'train_loss': 3.313075066263765, 'epoch': 5.0})

In [23]:
# CELL T2-9: Evaluate Task 2 RoBERTa + FE on test set

preds_t2 = trainer_t2.predict(test_enc_t2)

y_true_t2 = preds_t2.label_ids
y_pred_t2 = preds_t2.predictions.argmax(axis=-1)

print("=== Task 2 RoBERTa + FE – Test performance (FE_test_data) ===")
print(classification_report(
    y_true_t2,
    y_pred_t2,
    target_names=[id2label_t2[i] for i in range(num_labels_t2)],
))


=== Task 2 RoBERTa + FE – Test performance (FE_test_data) ===
                     precision    recall  f1-score   support

   Claims ignorance       0.31      0.38      0.34        24
      Clarification       0.78      0.39      0.52        18
Declining to answer       0.50      0.14      0.22        29
         Deflection       0.10      0.04      0.06        76
            Dodging       0.27      0.35      0.30       141
           Explicit       0.37      0.68      0.48       211
            General       0.00      0.00      0.00        77
           Implicit       0.14      0.06      0.08        98
Partial/half-answer       0.00      0.00      0.00        16

           accuracy                           0.32       690
          macro avg       0.27      0.23      0.22       690
       weighted avg       0.25      0.32      0.26       690



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
